# EXP-26: BSA–ΔG Correlation
**Linear regression of buried surface area vs binding free energy**

- **Expected:** Negative slope, R² > 0.5 (Janin 1997)
- **PASS:** R² > 0.5 AND slope in [−0.005, −0.020] kcal/mol/Å²
- **MARGINAL:** R² > 0.3 AND negative slope | **FAIL:** otherwise
- **Meta-analysis: loads results from prior experiments or uses expected values**

In [ ]:
!pip install -q scipy matplotlib numpy

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive', force_remount=True)
import sys
import json, zipfile, shutil
from pathlib import Path
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats

EXP_ID = 'EXP-26'
DRIVE_OUTPUT = Path(f'/content/drive/MyDrive/v3_gpu_results/{EXP_ID}')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path(f'/content/{EXP_ID}_work')
(WORK_DIR/'figures').mkdir(parents=True, exist_ok=True)

GPU_RESULTS = Path('/content/drive/MyDrive/v3_gpu_results')

# ── Console log capture ──────────────────────────────────────
import io as _io
_log_buf = _io.StringIO()
_orig_stdout_write = sys.stdout.write
_orig_stderr_write = sys.stderr.write
def _stdout_log_write(data):
    _orig_stdout_write(data)
    _log_buf.write(data)
def _stderr_log_write(data):
    _orig_stderr_write(data)
    _log_buf.write(data)
sys.stdout.write = _stdout_log_write
sys.stderr.write = _stderr_log_write
# ─────────────────────────────────────────────────────────────

In [ ]:
# Systems with known BSA and experimental \u0394G
systems = {
    'BPTI-Trypsin':    {'exp_id': 'EXP-04', 'bsa_exp': 1560, 'dg_exp': -18.0, 'pdb': '2PTC'},
    'PSTI-Chymo':      {'exp_id': 'EXP-05', 'bsa_exp': 1480, 'dg_exp': -14.7, 'pdb': '1TGS'},
    'SPINK1-Trypsin':  {'exp_id': 'EXP-06', 'bsa_exp': 1420, 'dg_exp': -11.1, 'pdb': '1TGS'},
    'Barnase-Barstar': {'exp_id': 'EXP-13', 'bsa_exp': 1590, 'dg_exp': -19.0, 'pdb': '1BRS'},
    'SH3-p41':         {'exp_id': 'EXP-29', 'bsa_exp': 950,  'dg_exp': -7.99, 'pdb': '4EIK'},
    'Eglin c-Subtilisin': {'exp_id': None,   'bsa_exp': 1350, 'dg_exp': -12.5, 'pdb': '1CSE'},
}

# Attempt to load computed values from prior experiments
for name, info in systems.items():
    if info['exp_id'] is None:
        info['dg_sim'] = info['dg_exp']  # fallback
        info['bsa_sim'] = info['bsa_exp']
        continue
    result_file = GPU_RESULTS / info['exp_id'] / 'results.json'
    if result_file.exists():
        with open(result_file) as f:
            res = json.load(f)
        info['dg_sim'] = res.get('delta_g_kcal_mol', res.get('delta_g_us_kcal', info['dg_exp']))
        info['bsa_sim'] = res.get('bsa_A2', info['bsa_exp'])
        print(f'  {name}: loaded from {info["exp_id"]} -> \u0394G={info["dg_sim"]:.1f}')
    else:
        info['dg_sim'] = info['dg_exp']
        info['bsa_sim'] = info['bsa_exp']
        print(f'  {name}: using expected values (result not found)')

bsa_vals = np.array([info['bsa_sim'] for info in systems.values()])
dg_vals = np.array([info['dg_sim'] for info in systems.values()])
print(f'\n{len(systems)} systems loaded')

In [ ]:
# Linear regression: \u0394G = slope * BSA + intercept
slope, intercept, r_value, p_value, std_err = stats.linregress(bsa_vals, dg_vals)
r_sq = r_value**2

print(f'Slope:     {slope:.5f} kcal/mol/\u00c5\u00b2')
print(f'Intercept: {intercept:.2f} kcal/mol')
print(f'R\u00b2:        {r_sq:.4f}')
print(f'p-value:   {p_value:.4e}')

# Verdict
if r_sq > 0.5 and -0.020 <= slope <= -0.005:
    verdict = 'PASS'
elif r_sq > 0.3 and slope < 0:
    verdict = 'MARGINAL'
else:
    verdict = 'FAIL'
print(f'\nVERDICT: {verdict}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(bsa_vals, dg_vals, s=80, c='steelblue', zorder=5, edgecolors='navy')
for i, (name, info) in enumerate(systems.items()):
    ax.annotate(name, (bsa_vals[i], dg_vals[i]), textcoords='offset points',
                xytext=(8, 5), fontsize=8)

x_fit = np.linspace(min(bsa_vals)-50, max(bsa_vals)+50, 100)
ax.plot(x_fit, slope*x_fit + intercept, 'r--', label=f'y={slope:.4f}x + {intercept:.1f} (R\u00b2={r_sq:.3f})')
ax.set_xlabel('Buried Surface Area (\u00c5\u00b2)'); ax.set_ylabel('\u0394G (kcal/mol)')
ax.set_title(f'EXP-26: BSA\u2013\u0394G Correlation \u2014 {verdict}')
ax.legend(); fig.tight_layout()
fig.savefig(WORK_DIR/'figures'/'results.png', dpi=150); plt.close(fig)

In [ ]:
results = {
    'experiment_id': EXP_ID, 'slope': round(float(slope), 6),
    'intercept': round(float(intercept), 2), 'r_squared': round(float(r_sq), 4),
    'p_value': float(p_value), 'n_systems': len(systems),
    'systems': {name: {'bsa': float(info['bsa_sim']), 'dg': float(info['dg_sim'])}
                for name, info in systems.items()},
    'verdict': verdict,
}
with open(WORK_DIR/'results.json', 'w') as f: json.dump(results, f, indent=2)

for item in WORK_DIR.rglob('*'):
    if item.is_file():
        dest = DRIVE_OUTPUT / item.relative_to(WORK_DIR)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, dest)
# ── Save full console log ────────────────────────────────────
_console_log_text = _log_buf.getvalue()
(WORK_DIR / 'console_log.txt').write_text(_console_log_text)
(DRIVE_OUTPUT / 'console_log.txt').write_text(_console_log_text)
print(f'Console log saved: {len(_console_log_text)} chars')
# ─────────────────────────────────────────────────────────────
zip_path = Path(f'/content/{EXP_ID}_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for item in WORK_DIR.rglob('*'):
        if item.is_file(): zf.write(item, f'{EXP_ID}/{item.relative_to(WORK_DIR)}')
files.download(str(zip_path))